# BASEUnet & CBAM Training Exp for DRIVE dataset (w halfaugs, newDataloader)

In [12]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import sys
sys.path.append('/content/drive/MyDrive/UNet Testing Template src/')
import src.training.metrics as m
print(m)

Mounted at /content/drive
<module 'src.training.metrics' from '/content/drive/MyDrive/UNet Testing Template src/src/training/metrics.py'>


In [13]:
from pathlib import Path
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# CHANGE THESE

In [14]:
TITLE = "BaseUNet & CBAM(yes aug; new dataloader; DRIVE)"

SAVE_FILE = "baseunet_cbam_w_halfaug_newDataloader_drive.pth"

SAVE_DIR = Path("/content/drive/MyDrive/UNet Testing Template src/pth")

DATA_DIR = "/content/drive/MyDrive/Retinal_Vessel_Segmentation_Datasets/DRIVE/"

LABEL_FOLDER = "1st_manual"


from src.models.ablations.unet_cbam import UNet as CBAMUNet

#pos_prior = p if 'p' in locals() else 0.10      # If you estimated 'p' earlier via estimate_class_weights, use it; else default to 0.10

model_core = CBAMUNet(
    in_ch=1,
    out_ch=1,
    reduction=16,
    use_spatial=True
).to(DEVICE)

# IMPORTS

In [15]:
import os, json, time, math, random
from pathlib import Path

import numpy as np
import torch.nn as nn
import torch.nn.functional as F

# --- data utils ---
from src.data.prepare_dataset import (
    build_pairs_for_split,
    build_all_train_pairs,
    assert_dataset_layout,
    sanity_check_sample_alignment,
)
from src.data.dataloader import make_loaders
from src.data.augmentations import get_train_augs, get_val_augs


# --- evaluation & visualization ---
from src.evaluation.evaluate import evaluate_and_print
from src.evaluation.visualization import visualize_samples

# SET SEED

In [16]:
SEED = 1337
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.benchmark = True

# MODEL PARAMS

In [17]:

# param count
print(sum(x.numel() for x in model_core.parameters() if x.requires_grad)/1e6, "M params")

31.261545 M params


# DATALOADERS

In [18]:
# 1) Pairs
train_pairs_full = build_pairs_for_split(DATA_DIR, split="training", label_folder=LABEL_FOLDER)   # 20 imgs
test_pairs       = build_pairs_for_split(DATA_DIR, split="test",     label_folder=LABEL_FOLDER)   # 20 imgs

# 2) Train/Val loaders: let make_loaders do an 80/20 split from training
#    (this is your "validation" during training; NOT the test set)
train_loader, val_loader = make_loaders(
    train_pairs=train_pairs_full,
    val_pairs=None,               # <-- auto 80/20 split from training
    image_size=512,
    batch_size=2,
    num_workers=1,                # Windows/Jupyter: keep 0
    seed=1337,
    strict_fov=True,
    augs_train=get_train_augs(512),
    augs_val=get_val_augs(512),   # deterministic, no random augs
)

# 3) Test loader: separate; same preprocessing as val (no random augs)
_, test_loader = make_loaders(
    train_pairs=test_pairs,       # dummy (ignored) – we just want the second loader built
    val_pairs=test_pairs,         # pass test_pairs so the returned "val" loader = test loader
    image_size=512,
    batch_size=2,
    num_workers=1,
    seed=1337,
    strict_fov=True,
    augs_train=None,              # unused
    augs_val=get_val_augs(512),   # no random augs
)

# LOSS

In [19]:
from src.training.loss_functions import DiceBCEComplementLoss
import torch

# (optional) estimate class weights from a few batches to handle heavy imbalance
def estimate_class_weights(loader, max_batches=20):
    with torch.no_grad():
        s = 0.0; n = 0
        for i, b in enumerate(loader):
            s += b["mask"].float().mean().item()  # foreground fraction in [0,1]
            n += 1
            if i+1 >= max_batches: break
    p = max(1e-6, min(1-1e-6, s / max(1, n)))      # foreground prior
    w1 = 1.0 / p                                    # weight FG inversely to its freq
    w0 = 1.0 / (1.0 - p)                            # weight BG inversely to its freq
    # normalize so w0 + w1 ≈ 2 (keeps scale stable)
    s2 = w0 + w1
    w0 = 2.0 * w0 / s2; w1 = 2.0 * w1 / s2
    return w0, w1, p

w0, w1, p = estimate_class_weights(train_loader, max_batches=20)
print(f"~ foreground prior p ≈ {p:.5f} | class weights -> w0={w0:.3f}, w1={w1:.3f}")

loss_fn = DiceBCEComplementLoss(
    w0=w0, w1=w1,
    dice_weight=0.5,         # tune (e.g., 0.6 Dice / 0.4 BCE)
    bce_weight=0.5,
    exact_equation=False,    # keep False so the BCE term is minimized (not added)
    reduction="mean",
)

~ foreground prior p ≈ 0.08617 | class weights -> w0=0.172, w1=1.828


# TRAIN LOOP

In [20]:
def train_one_epoch(model, loader, optimizer, scaler, loss_fn):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        x = batch["image"].to(DEVICE, non_blocking=True)  # [B,1,H,W]
        y = batch["mask"].to(DEVICE,  non_blocking=True)  # [B,1,H,W]

        # Optional: mask labels outside FOV if present (keeps loss fair)
        if "fov" in batch:
            fov = batch["fov"].to(DEVICE, non_blocking=True)
            y = y * (fov > 0.5).float()

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE=="cuda")):
            logits = model(x)          # ← PURE UNET: no fov passed
            loss   = loss_fn(logits, y)

        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total += loss.item() * x.size(0)
        n     += x.size(0)
    return total / max(1, n)

@torch.no_grad()
def validate_loss_only(model, loader, loss_fn):
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        x = batch["image"].to(DEVICE, non_blocking=True)
        y = batch["mask"].to(DEVICE,  non_blocking=True)

        if "fov" in batch:
            fov = batch["fov"].to(DEVICE, non_blocking=True)
            y = y * (fov > 0.5).float()

        with torch.amp.autocast(device_type="cuda", enabled=(DEVICE=="cuda")):
            logits = model(x)
            loss   = loss_fn(logits, y)

        total += loss.item() * x.size(0)
        n     += x.size(0)
    return total / max(1, n)

In [ ]:
import json, time, math
from torch.optim.lr_scheduler import LambdaLR

# --- optimizer / scaler / scheduler ---
optimizer = torch.optim.AdamW(model_core.parameters(), lr=3e-4, weight_decay=1e-4)
scaler    = torch.amp.GradScaler("cuda", enabled=(DEVICE=="cuda"))

EPOCHS = 100
warmup_epochs = 5

def lr_lambda(e):
    if e < warmup_epochs:
        return (e + 1) / warmup_epochs
    progress = (e - warmup_epochs) / max(1, EPOCHS - warmup_epochs)
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda)

best_val = float("inf")

# --- lists to collect losses for plotting later ---
train_losses, val_losses = [], []

for epoch in range(1, EPOCHS+1):
    print(f"----- EPOCH {epoch} -----")
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.time()

    tr_loss  = train_one_epoch(model_core, train_loader, optimizer, scaler, loss_fn)
    val_loss = validate_loss_only(model_core, val_loader, loss_fn)
    scheduler.step()

    if DEVICE == "cuda": torch.cuda.synchronize()
    print(f"[{epoch:03d}] train={tr_loss:.4f} val={val_loss:.4f} "
          f"lr={optimizer.param_groups[0]['lr']:.3e} time={(time.time()-t0):.1f}s")

    # record
    train_losses.append(float(tr_loss))
    val_losses.append(float(val_loss))

    # checkpoints
    if val_loss < best_val:
        best_val = val_loss
        torch.save(model_core.state_dict(), SAVE_DIR / SAVE_FILE)

----- EPOCH 1 -----
[001] train=1.5450 val=1.5651 lr=1.200e-04 time=42.8s
----- EPOCH 2 -----
[002] train=1.4357 val=1.5622 lr=1.800e-04 time=2.8s
----- EPOCH 3 -----
[003] train=1.3274 val=1.4832 lr=2.400e-04 time=2.7s
----- EPOCH 4 -----
[004] train=1.2097 val=1.3683 lr=3.000e-04 time=2.7s
----- EPOCH 5 -----
[005] train=1.1300 val=1.2834 lr=3.000e-04 time=2.6s
----- EPOCH 6 -----
[006] train=1.0851 val=1.1352 lr=2.999e-04 time=3.0s
----- EPOCH 7 -----
[007] train=1.0538 val=1.0775 lr=2.997e-04 time=2.7s
----- EPOCH 8 -----
[008] train=1.0234 val=1.0442 lr=2.993e-04 time=2.7s
----- EPOCH 9 -----
[009] train=1.0011 val=1.0509 lr=2.987e-04 time=2.8s
----- EPOCH 10 -----
[010] train=0.9798 val=1.0176 lr=2.980e-04 time=2.6s
----- EPOCH 11 -----
[011] train=0.9549 val=0.9969 lr=2.971e-04 time=3.0s
----- EPOCH 12 -----
[012] train=0.9344 val=0.9928 lr=2.960e-04 time=2.9s
----- EPOCH 13 -----
[013] train=0.9137 val=0.9781 lr=2.948e-04 time=2.8s
----- EPOCH 14 -----
[014] train=0.8909 val=0.

# PLOT

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,3.5))
plt.plot(range(1, len(train_losses)+1), train_losses, label="train")
plt.plot(range(1, len(val_losses)+1),   val_losses,   label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.grid(True); plt.legend()
plt.title("Train vs Val Loss:" + TITLE)
plt.tight_layout()
plt.show()

# METRICS EVAL

In [ ]:
# load best
model_core.load_state_dict(torch.load(SAVE_DIR / SAVE_FILE, map_location=DEVICE))
model_for_eval = model_core.eval()

# run your evaluation utility
print(TITLE)
evaluate_and_print(model_for_eval, test_dataloader=test_loader, device=DEVICE, threshold=0.5, compute_auc=True)

# VISUALIZATION

In [ ]:
visualize_samples(
    model=model_for_eval,
    dataloader=test_loader,
    n_rows=10,
    device=DEVICE,
    threshold=0.5,
    clamp_pred_with_fov=True,
    figsize_per_row=(12, 3),
)

# INFERENCE

In [ ]:
#@titleOne-click: upload image+mask → preprocess → infer → visualize (DRIVE)

# --- Colab / FS / imports ---
from google.colab import drive, files
drive.mount('/content/drive', force_remount=True)

import sys, os, cv2, time, math
import numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path

# ---- project path -----
sys.path.append('/content/drive/MyDrive/UNet Testing Template src/')

# ---- Model import & build (matches training config) ----
from src.models.wrappers.dpcn_concat_unet import DPCNConcatUNet

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMAGE_SIZE = 512  # must match what you trained on

CKPT_PATH = "/content/drive/MyDrive/UNet Testing Template src/pth/baseunet_dpcn_msu_cbam_hassskip_w_augs_newDataloader_drive_patching.pth"

BASE_KW = {"cbam_reduction": 16}
model = DPCNConcatUNet(
    in_ch=1,
    enh_channels=32,
    iters=4,
    threshold_mode="scaled_vat",
    half_life=2.0,
    reduce_to=64,
    base_kwargs=BASE_KW,
).to(DEVICE).eval()

# Load weights
print(f"Loading checkpoint from:\n  {CKPT_PATH}")
state = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(state)
print("✓ Model weights loaded.\n")

# ---- Minimal preprocessing helpers (green channel + iso-resize+pad + CLAHE + gamma) ----
def _iso_resize_and_pad(img: np.ndarray, target: int = 512, pad_value: float = 0):
    h, w = img.shape[:2]
    scale = float(target) / max(h, w)
    nh, nw = int(round(h*scale)), int(round(w*scale))
    interp = cv2.INTER_LINEAR if img.ndim == 3 else cv2.INTER_NEAREST
    resized = cv2.resize(img, (nw, nh), interpolation=interp)
    top = (target - nh) // 2; bottom = target - nh - top
    left = (target - nw) // 2; right = target - nw - left
    if img.ndim == 3:
        return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=[pad_value]*3)
    else:
        return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=pad_value)

def preprocess_image_retina(path: str, target_size=512, use_gamma=True, gamma=0.9, clahe_clip=2.0, clahe_tiles=8):
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None: raise FileNotFoundError(path)
    g_u8 = bgr[...,1]  # green channel
    g_u8 = _iso_resize_and_pad(g_u8, target=target_size, pad_value=0)
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=(clahe_tiles, clahe_tiles))
    g_eq = clahe.apply(g_u8)
    g = (g_eq.astype(np.float32) / 255.0)
    if use_gamma and 0.5 <= gamma <= 1.2:
        # simple gamma adjustment
        g = np.clip(g, 0, 1) ** gamma
    return np.expand_dims(g.astype(np.float32), axis=0)  # (1,H,W)

def preprocess_mask(path: str, target_size=512):
    m = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if m is None: raise FileNotFoundError(path)
    if m.ndim == 3: m = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)
    m = _iso_resize_and_pad(m, target=target_size, pad_value=0)
    # Otsu to {0,255} then -> {0,1}
    m = cv2.threshold(m, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)[1]
    m = (m > 0).astype(np.float32)
    return np.expand_dims(m, axis=0).astype(np.float32)   # (1,H,W)


In [ ]:
# ---- Upload image & GT ----
print("Upload the FUNDUS IMAGE file (png/jpg/tif)...")
im_up = files.upload()
img_path = next(iter(im_up.keys()))
print("Upload the corresponding GROUND-TRUTH MASK (binary, png/jpg/tif)...")
gt_up = files.upload()
gt_path = next(iter(gt_up.keys()))
print(f"\nImage: {img_path}\nMask : {gt_path}")

# ---- Preprocess ----
img_1hw = preprocess_image_retina(img_path, target_size=IMAGE_SIZE, use_gamma=True, gamma=0.9, clahe_clip=2.0, clahe_tiles=8)
gt_1hw  = preprocess_mask(gt_path, target_size=IMAGE_SIZE)

# FOV from padded zeros (safe default for single image)
fov_1hw = (img_1hw > 0).astype(np.float32)

# ---- To torch ----
x   = torch.from_numpy(img_1hw).unsqueeze(0).to(DEVICE)  # [1,1,H,W]
y   = torch.from_numpy(gt_1hw).unsqueeze(0).to(DEVICE)   # [1,1,H,W]
fov = torch.from_numpy(fov_1hw).unsqueeze(0).to(DEVICE)

# ---- Inference ----
t0 = time.time()
with torch.no_grad(), torch.amp.autocast(device_type="cuda", enabled=(DEVICE=="cuda")):
    logits = model(x, fov=fov)     # [1,1,H,W]
probs  = torch.sigmoid(logits)
pred01 = (probs >= 0.5).float()
dt = time.time()-t0
print(f"Inference time: {dt:.3f}s")

# ---- Metrics (Dice, IoU, AUC, + SEN, SPE, clDice) ----
def _flatten_masked(p, t, m=None):
    if m is None:
        return p.view(-1), t.view(-1)
    return (p*m).view(-1), (t*m).view(-1)

def dice_coefficient(p01, t01, m=None, eps=1e-6):
    p, t = _flatten_masked(p01, t01, m)
    tp = (p*t).sum()
    fp = (p*(1-t)).sum()
    fn = ((1-p)*t).sum()
    return (2*tp + eps) / (2*tp + fp + fn + eps)

def iou_score(p01, t01, m=None, eps=1e-6):
    p, t = _flatten_masked(p01, t01, m)
    tp = (p*t).sum()
    fp = (p*(1-t)).sum()
    fn = ((1-p)*t).sum()
    return (tp + eps) / (tp + fp + fn + eps)

def sensitivity_specificity(p01, t01, m=None, eps=1e-6):
    p, t = _flatten_masked(p01, t01, m)
    tp = (p*t).sum()
    tn = ((1-p)*(1-t)).sum()
    fp = (p*(1-t)).sum()
    fn = ((1-p)*t).sum()
    sen = (tp + eps) / (tp + fn + eps)  # recall
    spe = (tn + eps) / (tn + fp + eps)
    return sen, spe

# clDice requires skeletonization
import sys, subprocess
try:
    from skimage.morphology import skeletonize
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-image", "-q"])
    from skimage.morphology import skeletonize

def cldice_score(p01, t01, m=None, eps=1e-6):
    # convert to CPU numpy, apply mask, then skeletonize
    p = p01.detach().cpu().numpy().astype(np.uint8)[0,0]  # [H,W]
    g = t01.detach().cpu().numpy().astype(np.uint8)[0,0]
    if m is not None:
        mm = m.detach().cpu().numpy().astype(np.uint8)[0,0]
        p = (p * mm).astype(np.uint8)
        g = (g * mm).astype(np.uint8)
    sp = skeletonize(p > 0).astype(np.uint8)
    sg = skeletonize(g > 0).astype(np.uint8)
    # topology precision/recall
    tprec = (sp & g).sum() / (sp.sum() + eps)
    trec  = (sg & p).sum() / (sg.sum() + eps)
    return (2 * tprec * trec) / (tprec + trec + eps)

# compute all inside FOV
m = (fov > 0.5).float()

dice = dice_coefficient(pred01, y, m)
iou  = iou_score(pred01, y, m)
sen, spe = sensitivity_specificity(pred01, y, m)
cld  = cldice_score(pred01, y, m)

try:
    from sklearn.metrics import roc_auc_score
    auc = roc_auc_score(
        (y*m).detach().cpu().numpy().reshape(-1),
        (probs*m).detach().cpu().numpy().reshape(-1)
    )
except Exception:
    auc = None

print(
    "Dice: {:.4f} | IoU: {:.4f} | SEN: {:.4f} | SPE: {:.4f} | clDice: {:.4f}{}".format(
        dice.item(), iou.item(), sen.item(), spe.item(), cld, (f" | AUC: {auc:.4f}" if auc is not None else "")
    )
)

# ---- Viz ----
im  = x[0,0].detach().cpu().numpy()
gt  = y[0,0].detach().cpu().numpy()
pr  = probs[0,0].detach().cpu().numpy()
pb  = pred01[0,0].detach().cpu().numpy()

fig, axs = plt.subplots(1,4, figsize=(14,3.5))
axs[0].imshow(im, cmap='gray'); axs[0].set_title('Image'); axs[0].axis('off')
axs[1].imshow(gt, cmap='gray'); axs[1].set_title('GT'); axs[1].axis('off')
axs[2].imshow(pr, cmap='gray', vmin=0, vmax=1); axs[2].set_title('Prob'); axs[2].axis('off')
axs[3].imshow(pb, cmap='gray'); axs[3].set_title('Pred (τ=0.5)'); axs[3].axis('off')
plt.tight_layout(); plt.show()
